# Run a trained Relational Transformer on your own database

Turn your own relational database (here a small **DuckDB** store) into a
[RelBench](https://github.com/stanford-star/relbench) dataset, define tasks over it in SQL, and
predict them with the released **RT-J** checkpoint from HuggingFace, with no training.

*Runtime -> Change runtime type -> **A100** or **L4** GPU.* RT runs its attention in bf16,
which needs an Ampere-or-newer GPU. Colab's free T4 is older and cannot run it. A few
minutes end to end.

## 1. Install (one-time)

In [ ]:
# the repo is only needed for the demo database in notebooks/
import os

if os.path.exists("mini-shop.duckdb"):  # opened from a local clone's notebooks/
    %cd ..
elif not os.path.exists("notebooks/mini-shop.duckdb"):  # e.g. Colab: fetch the repo
    !git clone -q --depth 1 https://github.com/stanford-star/relational-transformer.git
    %cd relational-transformer
%pip install -q relational-transformer duckdb "relbench @ git+https://github.com/stanford-star/relbench@bac62c9"
print("setup done")

## 2. Your database as a RelBench dataset

`mini-shop.duckdb` is a toy store: `customers`, `products`, and a time-stamped
`transactions` log. A RelBench dataset is one Parquet file per table plus a `manifest.yaml`
naming each table's primary key, foreign keys and time column, and the val/test timestamps
that split it in time ([RelBench BYOD](https://github.com/stanford-star/relbench/blob/main/byod/README.md)).

In [ ]:
import duckdb
from relbench.manifest import DatasetManifest, TableSpec

R = "notebooks/mini-shop"
con = duckdb.connect("notebooks/mini-shop.duckdb", read_only=True)
os.makedirs(f"{R}/db", exist_ok=True)
for t in ["customers", "products", "transactions"]:
    con.execute(f"COPY {t} TO '{R}/db/{t}.parquet' (FORMAT PARQUET)")
DatasetManifest(
    name="mini-shop",
    val_timestamp="2021-10-18",
    test_timestamp="2021-11-15",
    tables={
        "customers": TableSpec(pkey="customer_id"),
        "products": TableSpec(pkey="product_id"),
        "transactions": TableSpec(
            pkey="transaction_id",
            time_col="timestamp",
            fkeys={"customer_id": "customers", "product_id": "products"},
        ),
    },
).save(f"{R}/manifest.yaml")

### Reading from a live SQL database (MySQL / Postgres)

DuckDB reads both directly, so swap the `duckdb.connect("…")` above for an `ATTACH` and export your own tables:

```python
con = duckdb.connect()                                   # in-memory
con.execute("INSTALL mysql; LOAD mysql")                 # Postgres: INSTALL postgres; LOAD postgres
con.execute("ATTACH 'host=… user=… password=… database=shop' AS shop (TYPE mysql)")
#                                                          Postgres: 'dbname=shop …' (TYPE postgres)
# then COPY shop.customers, shop.transactions, … TO parquet
```

RT only ever sees the exported Parquet, never your database. Check the keys: see *Use your own data* below.

## 3. Tasks in SQL

A RelBench `forecast` task is a SQL query: for each seed time in `timestamps` it emits
`(entity, timestamp, label)`, with the label computed from the `{timedelta}` window after
the timestamp, for entities with history before it. RelBench runs the query at the train,
val and test timestamps; we write the labels next to the task's `manifest.yaml`, where the
preprocessor reads them.

In [3]:
import pandas as pd
from relbench import load_dataset
from relbench.manifest import TaskManifest

CHURN = """
SELECT c.customer_id, s.timestamp, CAST(NOT EXISTS (
    SELECT 1 FROM transactions t WHERE t.customer_id = c.customer_id
    AND t.timestamp > s.timestamp AND t.timestamp <= s.timestamp + INTERVAL '{timedelta}'
) AS INTEGER) AS churn
FROM timestamps s, customers c
WHERE EXISTS (
    SELECT 1 FROM transactions t WHERE t.customer_id = c.customer_id AND t.timestamp < s.timestamp
)"""
SPEND = """
SELECT c.customer_id, s.timestamp, COALESCE((
    SELECT sum(t.amount) FROM transactions t WHERE t.customer_id = c.customer_id
    AND t.timestamp > s.timestamp AND t.timestamp <= s.timestamp + INTERVAL '{timedelta}'
), 0) AS spend
FROM timestamps s, customers c
WHERE EXISTS (
    SELECT 1 FROM transactions t WHERE t.customer_id = c.customer_id AND t.timestamp < s.timestamp
)"""
TASKS = [
    ("customer-churn", "churn", "binary_classification", CHURN),
    ("customer-spend", "spend", "regression", SPEND),
]

ds = load_dataset(R)
for name, target, task_type, sql in TASKS:
    d = f"{R}/tasks/{name}"
    TaskManifest(
        name=name, kind="forecast", task_type=task_type,
        entity_table="customers", entity_col="customer_id",
        target_col=target, time_col="timestamp", timedelta="28 days",
        sql=sql,
    ).save(f"{d}/manifest.yaml")
    task = ds.load_task(name, regenerate=True)
    db = task.get_db(upto_test_timestamp=False)
    for split in ["train", "val", "test"]:
        table = task.get_table(split, mask_input_cols=False, db=db)
        table.df.to_parquet(f"{d}/{split}.parquet", index=False)
pd.read_parquet(f"{R}/tasks/customer-churn/train.parquet").head()

,customer_id,timestamp,churn
0,0,2020-01-13,1
1,2,2020-01-13,0
2,6,2020-01-13,0
3,16,2020-01-13,0
4,20,2020-01-13,0


## 4. Preprocess

`preprocess` builds the on-disk format RT samples contexts from: the graph, then the text
embeddings.

In [ ]:
import sys  # noqa: F401  -- used by the !{sys.executable} shell escape below

!{sys.executable} -c "from rt.preprocess import one; one(dataset='notebooks/mini-shop', out_dir='notebooks/pre', embedder='all-MiniLM-L12-v2', batch_size=1024, skip_tasks=False, embed=True, upload_repo=None, public=False, revision=None)"

## 5. Predict and score

RT is an in-context learner, so a prediction needs a *context*: for each test row,
`build_evaluator` samples one (related rows from the database, the target masked), and the
model's forward over it is the prediction. One checkpoint handles classification and
regression alike. `run_and_report` maps each output back to its test row on the natural
scale (a probability, or the target's units), writes one CSV per task, and scores it with
RelBench.

> A locally trained `best_clf.safetensors` works as the checkpoint too.

In [5]:
import torch

from rt import RelationalTransformer
from rt.data import get_tasks, read_meta
from rt.eval import build_evaluator, run_and_report

print("GPU:", torch.cuda.get_device_name(0))
assert torch.cuda.get_device_capability()[0] >= 8, (
    f"{torch.cuda.get_device_name()} predates Ampere. RT's bf16 attention needs an "
    "A100, L4 or newer (on Colab: Runtime -> Change runtime type)"
)
model = RelationalTransformer.from_pretrained("stanford-star/rt-j", device="cuda").to(torch.bfloat16)
cfg = model.config  # dims + text-embedding model, read back from the checkpoint

pre_dir = "notebooks/pre"
pairs = [("mini-shop", t["name"]) for t in read_meta(pre_dir, "mini-shop")["tasks"]]
tasks = get_tasks(pre_dir, pairs, ("test",))
ev = build_evaluator(
    tasks, pre_dir,
    embedder=cfg["embedder"], d_text=cfg["d_text"], device="cuda",
    ctx_size_list=[8192], local_ctx_size=256, bfs_width=32, prefer_latest=True,
    num_walks=10_000, walk_length=20, tokens_per_gpu=2**16,
    items_per_task=10_000_000, num_workers=2, prefetch_factor=2,
    shuffle_seed=0, context_seed=0, mmap_populate=True, vector_db_path=None,
    db_cutoff=None,
)
run_and_report(
    model, tasks, pre_dir, ctx_size=8192, csv_out_dir="notebooks/preds",
    evaluator=ev, embedder=cfg["embedder"],
)
pd.read_csv("notebooks/preds/mini-shop__customer-churn.csv").head()

  task: mini-shop/customer-churn  metric: roc_auc  value: 0.7838  n: 1497  align: cls=1.000  norm_metric: roc_auc  norm_value: 0.7838


  task: mini-shop/customer-spend  metric: nmae     value: 0.5879  n: 1497  align: |dy|<=1.6e+00  norm_metric: mae      norm_value: 0.5887


,customer_id,timestamp,churn
0,1486,2021-11-15,0.396535
1,192,2021-11-15,0.570810
2,1346,2021-11-15,0.500437
3,560,2021-11-15,0.643329
4,1309,2021-11-15,0.616936


## Use your own data

Point step 2 at your own database (or `CREATE TABLE ... AS SELECT * FROM read_parquet('...')` /
`read_csv('...')`), name its keys and time columns, and pick the val/test timestamps. Rewrite
the task SQL in step 3: `task_type` is `binary_classification` or `regression`, and time
columns must be real datetimes. Steps 4-5 are unchanged and work for any number of tasks.

**Keys.** The tables must follow RelBench's [key contract](https://github.com/stanford-star/relbench/blob/main/byod/README.md#the-key-contract):
each primary key is `0..n-1` in row order (time-sorted if the table has a time column) and
foreign keys hold those row numbers. `mini-shop.duckdb` is built that way; auto-increment ids,
ids with gaps and string ids are not. Step 3 rejects primary keys that break it, and
`python byod/reindex_dataset.py notebooks/mini-shop` from a RelBench clone remaps the
exported tables in place, between steps 2 and 3.